In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from skpro.regression.linear import GLMRegressor

from pgmpy.base._base import _CoreGraph as BayesianNetwork
from pgmpy.parameter import LinearGaussianCPD, SklearnAdapter, SkproAdapter, TabularCPD
from pgmpy.parameter.BayesianLinearRegression import BayesianLinearRegression
from pgmpy.refactored_inference.LW import LikelihoodWeighting

import matplotlib.pyplot as plt
import pandas as pd
import pyro
import pyro.distributions as dist
import torch
from pyro.infer import SVI, Predictive, Trace_ELBO
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.nn import PyroModule, PyroSample
from skpro.distributions.normal import Normal as SkproNormal
from torch import nn

np.random.default_rng(42)


c:\Users\eogus\anaconda3\envs\pgmpy_311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Generator(PCG64) at 0x26F99E7EEA0

In [2]:
import numpy as np

rng = np.random.default_rng(42)
n_samples = 10_000

A = rng.binomial(
    n=1,
    p=0.35,
    size=n_samples,
)


# ------------------------------------------------------------
# C ~ Normal(0, 1)
# ------------------------------------------------------------

C = rng.normal(
    loc=0.0,
    scale=1.0,
    size=n_samples,
)


# ------------------------------------------------------------
# B | A ~ Normal(1 + 2.5 A, 0.5^2)
#
# A=0: B ~ Normal(1.0, 0.5^2)
# A=1: B ~ Normal(3.5, 0.5^2)
# ------------------------------------------------------------

B = (
    1.0
    + 2.5 * A
    + rng.normal(
        loc=0.0,
        scale=0.5,
        size=n_samples,
    )
)


# ------------------------------------------------------------
# P(D=1 | C) = sigmoid(-0.4 + 1.3 C)
# ------------------------------------------------------------

logit_D = -0.4 + 1.3 * C
prob_D = 1.0 / (1.0 + np.exp(-logit_D))

D = rng.binomial(
    n=1,
    p=prob_D,
    size=n_samples,
)


# ------------------------------------------------------------
# E | B,C ~ Normal(2 + 1.2 B - 0.8 C, 0.7^2)
# ------------------------------------------------------------

E = (
    2.0
    + 1.2 * B
    - 0.8 * C
    + rng.normal(
        loc=0.0,
        scale=0.7,
        size=n_samples,
    )
)


data = pd.DataFrame(
    {
        "A": A,
        "B": B,
        "C": C,
        "D": D,
        "E": E,
    }
)

print(data.head())
print(data.dtypes)


   A         B         C  D         E
0  1  4.100231 -1.883328  0  7.353651
1  0  1.176646  1.130478  1  1.276499
2  1  3.454701  1.182725  1  5.526219
3  1  4.367812 -0.474367  1  8.536282
4  0  1.007702  0.521276  0  1.930208
A      int64
B    float64
C    float64
D      int64
E    float64
dtype: object


In [3]:
bn = BayesianNetwork()
bn.SUPPORTED_EDGE_TYPES = frozenset(["->", "<-"])  # DAG

bn.add_edges_from(
    [
        ("A", "B", "->"),
        ("C", "D", "->"),
        ("B", "E", "->"),
        ("C", "E", "->"),
    ]
)

cpd_A = TabularCPD(categories={"A": [0, 1]})

cpd_B = SklearnAdapter(LinearRegression())

cpd_C = LinearGaussianCPD()

cpd_D = SklearnAdapter(
    LogisticRegression(
        max_iter=1_000,
        random_state=42,
    )
)

cpd_E = BayesianLinearRegression(
    in_features=2,
    num_iterations=500,
    lr=0.03,
    posterior_samples=100,
)
# cpd_E = SkproAdapter(
#     GLMRegressor(
#         family="Normal",
#         link="Identity",
#         add_constant=True,
#     )
# )
        

In [4]:
bn.add_cpd("A", cpd_A)
bn.add_cpd("B", cpd_B)
bn.add_cpd("C", cpd_C)
bn.add_cpd("D", cpd_D)
bn.add_cpd("E", cpd_E)


In [5]:
# P(A)
cpd_A.fit(data[["A"]])

# P(B | A)
cpd_B.fit(
    X=data[["A"]],
    y=data[["B"]],
)

# P(C)
cpd_C.fit(data[["C"]])

# P(D | C)
cpd_D.fit(
    X=data[["C"]],
    y=data[["D"]],
)

# P(E | B, C)
cpd_E.fit(
    X=data[["B", "C"]],
    y=data["E"],
)


c:\Users\eogus\anaconda3\envs\pgmpy_311\Lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


[iteration 0001] loss: 2.9319
[iteration 0101] loss: 1.0592
[iteration 0201] loss: 1.0681
[iteration 0301] loss: 1.0615
[iteration 0401] loss: 1.0601


BayesianLinearRegression(in_features=2, num_iterations=500,
                         posterior_samples=100)

In [6]:
parent_values = pd.DataFrame(
    {
        "A": [0, 1],
    }
)

b_dist = cpd_B.predict_proba(parent_values)

evidence_values = pd.DataFrame(
    {
        "B": [2.2, 2.2],
    }
)

b_likelihoods = np.asarray(b_dist.pdf(evidence_values)).reshape(-1)

print("p(B=2.2 | A=0):", b_likelihoods[0])
print("p(B=2.2 | A=1):", b_likelihoods[1])

assert np.all(np.isfinite(b_likelihoods))
assert np.all(b_likelihoods > 0)


p(B=2.2 | A=0): 0.04699942752087148
p(B=2.2 | A=1): 0.025380412056702766


In [ ]:
# n_samples=1000
# # SkproAdapter: 8.7[s]
# # BayesianLinearRegression: 142.5[s]

inference = LikelihoodWeighting()

posterior = inference.query(
    bn,
    variables=["A"],
    evidence={
        "B": 2.2,
    },
    n_samples=1000,
    seed=42,
)


In [11]:
posterior


Empirical(columns=Index(['A'], dtype='object'),
          index=Index([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,
       ...
       990, 991, 992, 993, 994, 995, 996, 997, 998, 999],
      dtype='int64', name='sample', length=1000),
          spl=                 A
instance sample   
0        0       1
         1       1
         2       0
         3       1
         4       0
...             ..
         995     0
         996     0
         997     0
         998     0
         999     1

[1000 rows x 1 columns],
          weights=instance  sample
0         0         0.000646
          1         0.000646
          2         0.001196
          3         0.000646
          4         0.001196
                      ...   
          995       0.001196
          996       0.001196
          997       0.001196
          998       0.001196
          999       0.000646
Length: 1000, dtype: float64)

In [12]:
posterior.spl.head()


A
instance sample   
0        0       1
         1       1
         2       0
         3       1
         4       0

In [13]:
posterior.weights.head()


instance  sample
0         0         0.000646
          1         0.000646
          2         0.001196
          3         0.000646
          4         0.001196
dtype: float64